# v2_02_train_plus : plus d'articles pour le TRAIN

Nouveau tirage de ~20 000 articles arXiv avec **les mêmes taux** que `v2_01_echantillon`, mais une **autre graine**.
Ils servent **uniquement à l'entraînement** : validation et test ne changent pas (comparaisons justes, pas de fuite).

- cellules 1 et 2 : identiques à `v2_01_echantillon` (table des groupes + comptage)
- cellule 3 : tirage en excluant les articles déjà dans train / val / test
- cellule 4 : même nettoyage que `02_pretraitement` + sauvegarde de `train_plus.csv`

In [ ]:
import glob, json, random
from collections import Counter
import pandas as pd
from tqdm.auto import tqdm

chemin = glob.glob("/kaggle/input/**/arxiv-metadata-oai-snapshot.json", recursive=True)[0]
print("Fichier :", chemin)

# Même table que dans la version 1 : archive arXiv -> un des 8 groupes
ARCHIVES_PAR_GROUPE = {
    "cs":      ["cs", "cmp-lg"],
    "math":    ["math", "q-alg", "alg-geom", "dg-ga", "funct-an"],
    "stat":    ["stat"],
    "eess":    ["eess"],
    "q-bio":   ["q-bio"],
    "q-fin":   ["q-fin"],
    "econ":    ["econ"],
    "physics": ["physics", "cond-mat", "astro-ph", "hep-ph", "hep-th", "hep-ex",
                "hep-lat", "quant-ph", "gr-qc", "math-ph", "nucl-th", "nucl-ex",
                "nlin", "chao-dyn", "solv-int", "patt-sol", "adap-org", "comp-gas",
                "mtrl-th", "supr-con", "chem-ph", "atom-ph", "acc-phys",
                "plasm-ph", "ao-sci", "bayes-an"],
}
GROUPE_PAR_ARCHIVE = {a: g for g, archives in ARCHIVES_PAR_GROUPE.items() for a in archives}
GROUPES = ["cs", "math", "physics", "q-bio", "q-fin", "stat", "eess", "econ"]

def categories_vers_groupes(categories):
    """Exemple : 'math.CO cs.CG' -> {'math', 'cs'}"""
    archives = {cat.split(".")[0] for cat in categories.split()}
    return {GROUPE_PAR_ARCHIVE[a] for a in archives}

print(categories_vers_groupes("cs.LG stat.ML"), categories_vers_groupes("hep-ph"))

In [ ]:
nb_articles = 0
compteur_groupes = Counter()
inconnues = Counter()

with open(chemin, "r") as f:
    for ligne in tqdm(f, desc="Comptage"):
        article = json.loads(ligne)
        nb_articles += 1
        archives = {cat.split(".")[0] for cat in article["categories"].split()}
        inconnues.update(a for a in archives if a not in GROUPE_PAR_ARCHIVE)
        compteur_groupes.update({GROUPE_PAR_ARCHIVE[a] for a in archives if a in GROUPE_PAR_ARCHIVE})

print(f"Nombre total d'articles : {nb_articles:,}")
print("Archives inconnues :", dict(inconnues) if inconnues else "aucune")
for g in GROUPES:
    print(f"  {g:8s} {compteur_groupes[g]:>10,}  ({100 * compteur_groupes[g] / nb_articles:.1f} %)")

In [ ]:
import os

# 1) Les articles déjà utilisés (train + val + test) : on ne doit PAS les retirer
DOSSIER = os.path.dirname(glob.glob("/kaggle/input/**/val.csv", recursive=True)[0])
deja = pd.concat([pd.read_csv(f"{DOSSIER}/{nom}.csv", dtype={"id": str}) for nom in ["train", "val", "test"]])
ids_deja = set(deja["id"])
print("Articles déjà utilisés (train + val + test) :", len(ids_deja))

# 2) Nouveau tirage : MÊMES taux que v2_01, mais une AUTRE graine -> d'autres articles
random.seed(2024)
TOTAL_VISE = 20_000
MINIMUM = 300
taux_base = TOTAL_VISE / nb_articles
taux = {g: max(taux_base, MINIMUM / compteur_groupes[g]) for g in GROUPES}

nouveaux = []
with open(chemin, "r") as f:
    for ligne in tqdm(f, desc="Tirage", mininterval=60):
        article = json.loads(ligne)
        groupes = categories_vers_groupes(article["categories"])
        if random.random() >= max(taux[g] for g in groupes):
            continue                      # pas tiré
        if article["id"] in ids_deja:
            continue                      # déjà dans train/val/test
        if len(article["abstract"].split()) < 30:
            continue                      # résumé trop court
        resultat = {"id": article["id"], "title": article["title"], "abstract": article["abstract"]}
        for g in GROUPES:
            resultat[g] = 1 if g in groupes else 0
        nouveaux.append(resultat)

plus = pd.DataFrame(nouveaux)
print("Nouveaux articles tirés :", len(plus))

In [ ]:
import re

# Même nettoyage que 02_pretraitement
def nettoyer(texte):
    texte = texte.replace(r"\$", " dollars ")                            # \$100 -> dollars 100
    texte = re.sub(r"\$\$.*?\$\$", " formula ", texte, flags=re.DOTALL)  # $$ ... $$ -> formula
    texte = re.sub(r"\$.*?\$", " formula ", texte, flags=re.DOTALL)      # $ ... $   -> formula
    texte = re.sub(r"\\[a-zA-Z]+", " ", texte)                           # \emph, \cite... -> supprimés
    texte = re.sub(r"\\[^a-zA-Z\s]", "", texte)                          # accents : \" \' -> supprimés
    texte = texte.replace("\\", " ")                                     # backslashs restants
    texte = re.sub(r"[{}]", "", texte)                                   # accolades { }
    texte = re.sub(r"\s+", " ", texte)                                   # \n et espaces multiples -> 1 espace
    return texte.strip()                                                 # espaces au début et à la fin

plus["title"] = plus["title"].apply(nettoyer)
plus["text"] = plus["title"].str.rstrip(" .") + ". " + plus["abstract"].apply(nettoyer)

# Sécurité anti-fuite : aucun texte identique à un texte déjà utilisé, pas de doublons
avant = len(plus)
plus = plus[~plus["text"].isin(set(deja["text"]))].drop_duplicates(subset="text")
print("Retirés (texte déjà vu) :", avant - len(plus))
print("Ids en commun avec train/val/test :", len(set(plus["id"]) & ids_deja), "(doit être 0)")
print("Backslashs restants :", plus["text"].str.contains("\\", regex=False).sum())
print(pd.DataFrame({"nombre": plus[GROUPES].sum(),
                    "%": (100 * plus[GROUPES].mean()).round(1)}).sort_values("nombre", ascending=False))
print("Groupes par article en moyenne :", round(plus[GROUPES].sum(axis=1).mean(), 2))

COLONNES = ["id", "title", "text"] + GROUPES
plus[COLONNES].to_csv("/kaggle/working/train_plus.csv", index=False)
print("Fichier sauvegardé : /kaggle/working/train_plus.csv")